#  1 - Importing Necessary Libraries

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 2 - Data Cleaning and Preparation

In [3]:
column_names = ['fLength', 'fWidth', 'fSize', 'fConc', 'fConc1',
                'fAsym', 'fM3Long', 'fM3Trans', 'fAlpha', 'fDist', 'class']

data = pd.read_csv('magic04.data',names = column_names)

In [4]:
print(data.isnull().sum())

fLength     0
fWidth      0
fSize       0
fConc       0
fConc1      0
fAsym       0
fM3Long     0
fM3Trans    0
fAlpha      0
fDist       0
class       0
dtype: int64


## 2.1 Convert class labels to numeric

In [5]:
class_mapping = {'g': 1, 'h': 0} 
data['class'] = data['class'].map(class_mapping)

## 2.2 Data Balancing

In [6]:
class_0 = data[data['class'] == 0] 
class_1 = data[data['class'] == 1]

# Undersample the majority class (class 1) to match the size of the minority class (class 0)
class_1_undersampled = class_1.sample(len(class_0), random_state=42)

# Combine the minority class and the undersampled majority class
balanced_df = pd.concat([class_0, class_1_undersampled], axis=0)

# Shuffle the balanced dataset
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nBalanced dataset shape:", balanced_df.shape)
print("\nBalanced class distribution (Counts):")
print(balanced_df['class'].value_counts())


Balanced dataset shape: (13376, 11)

Balanced class distribution (Counts):
class
1    6688
0    6688
Name: count, dtype: int64


## 2.3 Data Splitting

In [7]:
X = balanced_df.drop(columns=['class'])
y = balanced_df['class']

# Split into 70% training and 30% testing
# Using stratify ensures the class proportion is maintained in train/test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"\nX_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")


X_train shape: (9363, 10)
y_train shape: (9363,)
X_test shape: (4013, 10)
y_test shape: (4013,)


## 2.4 Feature Scaling

In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nX_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")


X_train_scaled shape: (9363, 10)
X_test_scaled shape: (4013, 10)


# Naive Bayes Classification Model

This code cell implements a Gaussian Naive Bayes classifier for a machine learning classification task. Here's what the code does:

1. **Imports necessary libraries**:
   - `GaussianNB` from scikit-learn's naive_bayes module
   - Classification evaluation metrics from scikit-learn

2. **Creates and trains the model**:
   - Initializes a Gaussian Naive Bayes classifier
   - Fits the model using the scaled training data (`X_train_scaled` and `y_train`)

3. **Makes predictions**:
   - Uses the trained model to predict classes for the scaled test data

4. **Evaluates model performance**:
   - Generates a classification report showing precision, recall, f1-score, and support
   - Creates a confusion matrix to visualize prediction errors
   - Prints both evaluation metrics to assess model performance

In [9]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report,confusion_matrix

model=GaussianNB()
model.fit(X_train_scaled,y_train)

y_predict=model.predict(X_test_scaled)

report=classification_report(y_test,y_predict)
matrix=confusion_matrix(y_test,y_predict)
print("Classification Report:\n",report)
print("Confusion Matrix:\n",matrix)

Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.39      0.52      2007
           1       0.59      0.90      0.72      2006

    accuracy                           0.64      4013
   macro avg       0.69      0.64      0.62      4013
weighted avg       0.69      0.64      0.62      4013

Confusion Matrix:
 [[ 781 1226]
 [ 205 1801]]


# 3 - Decision Tree Implementation

**Logic:**
1.  **Instantiate:** Create an instance of `DecisionTreeClassifier`. We use `random_state=42` for reproducibility.
2.  **Train:** Fit the model to the scaled training data (`X_train_scaled`, `y_train`). The model learns decision rules by recursively splitting the data based on features that best separate the classes (typically maximizing information gain or minimizing Gini impurity).
3.  **Predict:** Use the trained model to predict class labels for the unseen scaled test data (`X_test_scaled`).
4.  **Evaluate:** Compare the predicted labels (`y_pred_dt`) with the true labels (`y_test`) using standard classification metrics.

In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- Decision Tree ---

# 1. Instantiate the Decision Tree Classifier
dt_model = DecisionTreeClassifier(random_state=42)

# 2. Train the model
print("Training Decision Tree model...")
dt_model.fit(X_train_scaled, y_train)
print("Training complete.")

# 3. Make predictions on the scaled test set
y_pred_dt = dt_model.predict(X_test_scaled)

# 4. Evaluate the model
print("\nDecision Tree - Evaluation on Test Set:")
dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_precision = precision_score(y_test, y_pred_dt)
dt_recall = recall_score(y_test, y_pred_dt)
dt_f1 = f1_score(y_test, y_pred_dt)
dt_cm = confusion_matrix(y_test, y_pred_dt)

print(f"Accuracy:  {dt_accuracy:.4f}")
print(f"Precision: {dt_precision:.4f}")
print(f"Recall:    {dt_recall:.4f}")
print(f"F1-Score:  {dt_f1:.4f}")
print("Confusion Matrix:")
print(dt_cm)


print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt, target_names=['Hadron (0)', 'Gamma (1)']))

Training Decision Tree model...
Training complete.

Decision Tree - Evaluation on Test Set:
Accuracy:  0.7869
Precision: 0.7853
Recall:    0.7896
F1-Score:  0.7875
Confusion Matrix:
[[1574  433]
 [ 422 1584]]

Classification Report:
              precision    recall  f1-score   support

  Hadron (0)       0.79      0.78      0.79      2007
   Gamma (1)       0.79      0.79      0.79      2006

    accuracy                           0.79      4013
   macro avg       0.79      0.79      0.79      4013
weighted avg       0.79      0.79      0.79      4013



# 4 - Random Forest Implementation

Next, we implement the Random Forest classifier. Random Forest is an ensemble method that builds multiple decision trees during training and outputs the mode of the classes (classification) of the individual trees. It generally improves upon a single decision tree by reducing overfitting.

The assignment requires tuning the `n_estimators` parameter, which controls the number of trees in the forest. We will use `GridSearchCV` for this purpose.

**Logic:**
1.  **Instantiate Base Model:** Create an instance of `RandomForestClassifier`.
2.  **Define Parameter Grid:** Specify a range of values for `n_estimators` to test.
3.  **Setup GridSearchCV:** Configure `GridSearchCV` to use the Random Forest model, the parameter grid, 5-fold cross-validation (`cv=5`), and accuracy as the scoring metric. `n_jobs=-1` uses all available CPU cores.
4.  **Tune Parameters:** Fit `GridSearchCV` to the scaled training data. It will train and evaluate models for each combination of parameters using cross-validation.
5.  **Get Best Model:** Retrieve the best estimator (model trained with the optimal `n_estimators`) found by `GridSearchCV`.
6.  **Predict:** Use the best tuned model to predict class labels for the scaled test data.
7.  **Evaluate:** Compare the predictions with the true labels using the standard metrics.

In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# --- Random Forest ---

# 1. Instantiate the base Random Forest Classifier
rf_base = RandomForestClassifier(random_state=42)

# 2. Define the parameter grid for n_estimators
param_grid_rf = {
    'n_estimators': [50, 100, 150, 200, 250] # Values to test as per assignment example
}

# 3. Set up GridSearchCV
print("\nPerforming GridSearchCV for Random Forest n_estimators...")
grid_search_rf = GridSearchCV(estimator=rf_base,
                              param_grid=param_grid_rf,
                              cv=5, # 5-fold cross-validation
                              scoring='accuracy',
                              n_jobs=-1, # Use all available cores
                              verbose=1) # Show progress

# 4. Fit GridSearchCV on the scaled training data
grid_search_rf.fit(X_train_scaled, y_train)
print("GridSearchCV complete.")

# 5. Get the best estimator and parameters
best_rf_model = grid_search_rf.best_estimator_
best_n_estimators_rf = grid_search_rf.best_params_['n_estimators']

print(f"\nBest n_estimators found for Random Forest: {best_n_estimators_rf}")
print(f"Best cross-validation accuracy: {grid_search_rf.best_score_:.4f}")

# 6. Make predictions using the best model on the scaled test set
y_pred_rf = best_rf_model.predict(X_test_scaled)

# 7. Evaluate the best Random Forest model
print("\nRandom Forest (Best Tuned Model) - Evaluation on Test Set:")
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
rf_cm = confusion_matrix(y_test, y_pred_rf)

print(f"Accuracy:  {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall:    {rf_recall:.4f}")
print(f"F1-Score:  {rf_f1:.4f}")
print("Confusion Matrix:")
print(rf_cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Hadron (0)', 'Gamma (1)']))


Performing GridSearchCV for Random Forest n_estimators...
Fitting 5 folds for each of 5 candidates, totalling 25 fits
GridSearchCV complete.

Best n_estimators found for Random Forest: 200
Best cross-validation accuracy: 0.8541

Random Forest (Best Tuned Model) - Evaluation on Test Set:
Accuracy:  0.8610
Precision: 0.8371
Recall:    0.8963
F1-Score:  0.8657
Confusion Matrix:
[[1657  350]
 [ 208 1798]]

Classification Report:
              precision    recall  f1-score   support

  Hadron (0)       0.89      0.83      0.86      2007
   Gamma (1)       0.84      0.90      0.87      2006

    accuracy                           0.86      4013
   macro avg       0.86      0.86      0.86      4013
weighted avg       0.86      0.86      0.86      4013

